# TWO-STAGE MODEL 

In [ ]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import mlflow
from statsmodels.tsa.stattools import pacf
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from imblearn.over_sampling import SMOTE
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import  pickle
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


df = pd.read_csv("./ElectronicsProductsPricingData.csv", encoding='latin1')



df["prices.dateSeen"] = pd.to_datetime(df["prices.dateSeen"], errors='coerce')


# compute a percent‑discount column from the price fields
# y = ((regular_price - sale_price) / regular_price) * 100
# assume amountMax is the non‑sale price and amountMin the current price;
# guard against division by zero.

df['y'] = np.where(
    df['prices.amountMax'] > 0,
    100 * (df['prices.amountMax'] - df['prices.amountMin']) / df['prices.amountMax'],
    0
)
mlflow.end_run()
df = df.rename(columns={"prices.dateSeen": "date", "y": "y"})
df[["y","prices.amountMax","date","prices.amountMin"]].head()
mlflow.set_tracking_uri("http://86.50.20.250/")



In [20]:
from scripts.cleaning_categories import clean_and_map_categories

# ---- 4. Apply to dataframe ----
df["clean_categories"] = df["categories"].apply(clean_and_map_categories)

# ---- 5. Create MAIN main_main_category (for plotting) ----
def get_main_category(cat_list):
    return cat_list[0] if len(cat_list) > 0 else "unknown"

df["main_category"] = df["clean_categories"].apply(get_main_category)

# ---- 6. Optional: string version (good for plotting labels) ----
df["clean_categories_str"] = df["clean_categories"].apply(lambda x: " | ".join(x))

## Featureing enginnering 

In [21]:
df_2018 = df[df["date"].dt.year == 2018]
df = df[df["date"].dt.year == 2017]

train = df
test = df_2018


In [ ]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

df_2018['day_of_week'] = df_2018['date'].dt.dayofweek
df_2018['month'] = df_2018['date'].dt.month
df_2018['week_of_year'] = df_2018['date'].dt.isocalendar().week.astype(int)


# Binary flag for Thursday updates
df['is_thursday'] = (df['day_of_week'] == 3).astype(int)
df_2018['is_thursday'] = (df_2018['day_of_week'] == 3).astype(int)

### Our seasonality chart showed Q4  = hilday dicsount spike in 2017 

df['is_q4'] = df['month'].isin([10, 11, 12]).astype(int)
df_2018['is_q4'] = df_2018['month'].isin([10, 11, 12]).astype(int)


df["price_tier"] = pd.cut(df["prices.amountMin"], bins=[0, 50, 100, 200, np.inf], labels=['Low', 'Medium', 'High', 'Premium'])
df_2018["price_tier"] = pd.cut(df_2018["prices.amountMin"], bins=[0, 50, 100, 200, np.inf], labels=['Low', 'Medium', 'High', 'Premium'])



# Preview the engineered features
df[['date', 'day_of_week', 'week_of_year', 'is_thursday']].head()

In [ ]:
train_orig = train.copy()
test_orig  = test.copy()

# ── Binary flag: 1 = non-zero, 0 = zero ──
train_orig["is_nonzero"] = (train_orig["y"] > 0).astype(int)
test_orig["is_nonzero"]  = (test_orig["y"]  > 0).astype(int)

print("Train zero %    :", (train_orig["is_nonzero"] == 0).sum() / len(train_orig) * 100)
print("Test  zero %    :", (test_orig["is_nonzero"]  == 0).sum() / len(test_orig)  * 100)
print("Non-zero count  :", train_orig["is_nonzero"].sum())

In [ ]:
train_copy = test_orig[test_orig["is_nonzero"] == 1].copy() 
train_copy = train_copy.groupby("main_category").count().sort_values(by="main_category", ascending=False)
train_copy.head(5)



### STAGE 1 - Logistics Regressions
- 

In [ ]:
def make_lag_features(series, lags=[1, 2, 3, 7, 14, 30]):
    df = pd.DataFrame({"y": series})
    for lag in lags:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    df["rolling_mean_3"]  = df["y"].shift(1).rolling(3).mean()
    df["rolling_mean_7"]  = df["y"].shift(1).rolling(7).mean()
    df["rolling_mean_14"] = df["y"].shift(1).rolling(14).mean()

    # ---- TIME FEATURES ----
    df["is_thursday"] = (pd.to_datetime(series.index).dayofweek == 3).astype(int)
    df["is_q4"]       = (pd.to_datetime(series.index).quarter   == 4).astype(int)

    # ---- CATEGORY FEATURES ----
    source = train_orig if len(series) == len(train_orig) else test_orig

    df["is_tv_video"] = source["main_category"].str.lower().isin(
                            ["tv", "video", "tv & video"]
                        ).astype(int).values[:len(df)]
    

    df["is_computer"] = source["main_category"].str.lower().isin(
                            ["computer", "computers", "computing"]
                        ).astype(int).values[:len(df)]

    cat_zero_rate = (train_orig.groupby("main_category")["y"]
                     .apply(lambda x: (x == 0).mean()))
    df["cat_zero_rate"] = (source["main_category"]
                           .map(cat_zero_rate)
                           .values[:len(df)])
    cat_mean = train_orig.groupby("main_category")["y"].mean()
    df["cat_mean_discount"] = (source["main_category"]
                                .map(cat_mean)
                                .values[:len(df)])

    

    

    

    return df.dropna()


# ── Rebuild features ──
train_binary = make_lag_features(train_orig["is_nonzero"])
test_binary  = make_lag_features(test_orig["is_nonzero"])

feature_cols = [c for c in train_binary.columns if c != "y"]

X_train_clf = train_binary[feature_cols]
y_train_clf = train_binary["y"]
X_test_clf  = test_binary[feature_cols]
y_test_clf  = test_binary["y"]

# ── Scale + Fit ──
scaler = StandardScaler()
X_train_clf_sc = scaler.fit_transform(X_train_clf)
X_test_clf_sc  = scaler.transform(X_test_clf)

clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(X_train_clf_sc, y_train_clf)

# ── Calculate actual class weight from data ──
zero_count    = (y_train_clf == 0).sum()
nonzero_count = (y_train_clf == 1).sum()
total         = len(y_train_clf)

# Natural weight — reflects true sparsity
weight_0 = 1.0
weight_1 = zero_count / nonzero_count   # e.g. if 90% zeros → weight_1 = 9

print(f"Zero count    : {zero_count}")
print(f"Non-zero count: {nonzero_count}")
print(f"Weight for 1  : {weight_1:.2f}")

clf = LogisticRegression(
    class_weight={0: weight_0, 1: weight_1 * 0.5},  # ✅ dampen the minority boost
    max_iter=1000,
    C=0.1          # ✅ stronger regularization — reduces over-prediction
)
clf.fit(X_train_clf_sc, y_train_clf)

zero_pred       = clf.predict(X_test_clf_sc)
zero_pred_proba = clf.predict_proba(X_test_clf_sc)[:, 1]

print(f"\nPredicted non-zero: {zero_pred.sum()} / {len(zero_pred)}")
print(f"Actual  non-zero  : {int(y_test_clf.sum())} / {len(y_test_clf)}")

In [ ]:
# ── Extract only non-zero rows ──
train_nonzero = train_orig[train_orig["y"] > 0].copy()

# ── Log transform ──
train_nonzero["y_log"] = np.log1p(train_nonzero["y"])

# ── Fit best SARIMA on non-zero data ──
best_rmse_s   = float("inf")
best_aic_s    = float("inf")
best_sarima   = None
best_params_s = None
best_model = None
m = 12

for p in range(0, 3):
    for q in range(0, 3):
        for P in range(0, 2):
            for Q in range(0, 7):
                

                if p == 0 and q == 0 and P == 0 and Q == 0:
                    continue

                try:
                    model = SARIMAX(
                        train_nonzero["y_log"],
                        order=(p, 0, q),
                        seasonal_order=(P, 1, Q, m),
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    )
                    model_fit = model.fit(disp=False)
                    aic = model_fit.aic

                    if aic < best_aic_s:
                        best_aic_s    = aic
                        best_sarima   = model_fit
                        best_params_s = (p, 0, q, P, 1, Q, m)
                        best_model = model_fit
                except Exception as e:
                    continue

print(f"\n✅ Best SARIMA params : {best_params_s}")
print(f"   Best AIC          : {best_aic_s:.4f}")

In [ ]:
# ── SARIMA forecasts (log scale → reverse) ──
n_test = len(test_orig)

sarima_forecast_log = best_sarima.forecast(n_test)
sarima_forecast     = np.expm1(sarima_forecast_log)   # back to original scale

# ── Align lengths (lag features drop some rows) ──
offset = n_test - len(zero_pred)   # rows lost due to lag creation
sarima_aligned = sarima_forecast[offset:]
actual_aligned = test_orig["y"].values[offset:]

# ── Combine: zero_pred gates the sarima value ──
# Method 1 — Hard gate (0 or full value)
final_forecast_hard = zero_pred * sarima_aligned

# Method 2 — Soft gate (scale by probability)
final_forecast_soft = zero_pred_proba * sarima_aligned

print("\nForecast preview:")
preview = pd.DataFrame({
    "Actual"        : actual_aligned[:10],
    "SARIMA"        : sarima_aligned[:10].round(4),
    "Zero_Pred"     : zero_pred[:10],
    "Final_Hard"    : final_forecast_hard[:10].round(4),
    "Final_Soft"    : final_forecast_soft[:10].round(4),
})
print(preview.to_string())

In [ ]:
def evaluate(actual, forecast, label):
    rmse = np.sqrt(mean_squared_error(actual, forecast))
    mae  = np.mean(np.abs(actual - forecast))
    # RMSE % relative to actual mean
    rmse_pct = (rmse / actual.mean()) * 100 if actual.mean() != 0 else float("inf")

    print(f"\n  [{label}]")
    print(f"    RMSE      : {rmse:.4f}")
    print(f"    MAE       : {mae:.4f}")
    print(f"    RMSE %    : {rmse_pct:.2f}%")
    return rmse

print("\n===== TWO-STAGE MODEL EVALUATION =====")
rmse_hard = evaluate(actual_aligned, final_forecast_hard, "Hard Gate (0 or value)")
rmse_soft = evaluate(actual_aligned, final_forecast_soft, "Soft Gate (prob × value)")
rmse_base = evaluate(actual_aligned, sarima_aligned,      "SARIMA alone (baseline)")

print(f"\n  Best method: {'Hard' if rmse_hard < rmse_soft else 'Soft'} gate")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# ── Plot 1: Full comparison ──
axes[0].plot(actual_aligned,        label="Actual",       color="green",  linewidth=1.5)
axes[0].plot(sarima_aligned,        label="SARIMA alone", color="orange", linewidth=1,  linestyle="--")
axes[0].plot(final_forecast_hard,   label="Two-Stage",    color="blue",   linewidth=1.5, linestyle="-.")
axes[0].set_title("Two-Stage Model vs SARIMA Alone vs Actual")
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Plot 2: Zero prediction accuracy ──
axes[1].plot((actual_aligned > 0).astype(int), label="Actual Non-Zero",    color="green")
axes[1].plot(zero_pred,                         label="Predicted Non-Zero", color="red", linestyle="--")
axes[1].set_title("Stage 1 — Zero/Non-Zero Classification")
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── Plot 3: Residuals ──
residuals = actual_aligned - final_forecast_hard
axes[2].plot(residuals, color="red", linewidth=0.8)
axes[2].axhline(0, color="black", linestyle="--")
axes[2].set_title(f"Residuals (Two-Stage)  |  RMSE: {rmse_hard:.4f}")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("./images/two_stage_forecast.png", dpi=150)
plt.show()

In [ ]:
from scripts.testing import check_two_stage_performance

check_two_stage_performance(
    clf             = clf,
    regressor       = best_model,
    train           = train,
    test            = test,
    final_forecast  = final_forecast_hard,
    zero_pred       = zero_pred,
    zero_pred_proba = zero_pred_proba,
    sarima_forecast = sarima_aligned,
    best_params     = best_params_s
)

In [ ]:
from scripts.testing import compare_models

compare_models(
    actual             = test["y"].values,
    sarima_forecast    = sarima_aligned,
    two_stage_forecast = final_forecast_hard
)